# Chapter 7: Adversarial Test Discovery for Agentic Systems

An agentic validation system must resist inputs designed to make it report something false. Chapter 7 constructs such tests by design: attack factors in an Inner Array, operating conditions in an Outer Array, a declared expected outcome for every crossing and a search for the combinations that produce failures. This notebook applies the crossing to the deterministic claim verifier of Chapter 6, using the regional evidence of the medoid search.

The notebook fixes two reference cases, constructs five claim mutations with declared expected verdicts, crosses and scores them and summarizes the design with its limits. It makes no live language-model call. The chapter's live experiments are recorded in other notebooks named in the last section.

**Evidence.** The store is `book/evidence/parallel_aml_20260921/store.json`, whose regional records come from the run used in Chapters 1, 3 and 4. The population is synthetic.

**Run order.** Run the cells from top to bottom. The crossing asserts that every verdict equals its declared expectation.

## Setup and source checks

Use a Python kernel with the dependencies listed in the README. The setup cell locates the data root, which holds the evidence files and helper modules. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package.

`load()` checks the digests that link the regional results, frozen finalists, protocol and admission records. The verifier is imported from `book/parallel_evidence/build.py` under the data root.

In [1]:
from pathlib import Path
import os
import platform
import sys
import importlib.util

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
HELPERS = ROOT / "book/notebooks/regional_case"
for path in (ROOT, ROOT / "book", HELPERS):
    value = str(path)
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

# Load current helper files even if this kernel cached an older notebook revision.
def load_helper_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_helper_module("companion", HELPERS / "companion.py")
load_helper_module("ch02_helpers", HELPERS / "ch02_helpers.py")
load_helper_module("weighted_leaf_geometry", ROOT / "book/weighted_leaf_geometry.py")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
from companion import RUN, read, load, frontier, confirmation, measure, assignments
pd.set_option("display.float_format", "{:.6f}".format)
protocol, results, frozen = load()
print("Evidence:", RUN.relative_to(ROOT))
print("Mode: saved-evidence replay; no fitting or live LLM calls")


Python: 3.12.3
Data root found from: repository checkout
Evidence: book/evidence/aml_medoid_loop_leaf_output_20260921
Mode: saved-evidence replay; no fitting or live LLM calls


## Fix the reference cases

Chapter 7 designs adversarial tests for an agentic system by separating two roles (Section 7.2). Attack factors, the ways an input is altered to provoke a failure, form the Inner Array of candidate tests. Operating conditions, the contexts in which each attack is applied, form the Outer Array. Crossing the two shows whether a failure depends on the attack, on the condition or on both.

This notebook applies that design to a smaller object: the deterministic verifier of Chapter 6, which a reporting agent would use to check its numbers. The tested failure is a false accept, a verdict of Supported for a claim that misstates the evidence. The system is deterministic and no language model is called, so the design tests the verifier's contract under constructed inputs; it does not measure how an agent responds to misleading prompts.

The Outer Array holds two reference cases, two separately scoped measurements of the same frozen region: the confirmation Brier loss and the confirmation contrast of the seed-73101 reflection region from Chapter 4. The table shows each case's value, source file and pointer.

In [2]:
from book.parallel_evidence.build import OUT, verify
store=read(OUT/"store.json")
outer=["region:confirmation/reflection_73101/brier","region:confirmation/reflection_73101/contrast"]
display(pd.DataFrame([dict(case=t,value=store["facts"][t]["value"],source=store["facts"][t]["source"],
                           pointer=store["facts"][t]["pointer"]) for t in outer]))

,case,value,source,pointer
0,region:confirmation/reflection_73101/brier,0.072851,book/evidence/aml_medoid_loop_leaf_output_2026...,/confirmation/reflection_73101/brier
1,region:confirmation/reflection_73101/contrast,0.058326,book/evidence/aml_medoid_loop_leaf_output_2026...,/confirmation/reflection_73101/contrast


**Read the output.** The two reference cases share a region and a split but differ in metric: Brier loss 0.072851 and contrast 0.058326, both read from the results file at their own pointers. Because the values differ and the addresses differ only in the final field, the pair can reveal whether a verifier binds a value to its metric or accepts any recorded number.

## Construct the mutations

Each Inner Array row is an attack that transforms a faithful claim into an altered one. The expected verdict of each attack is declared before any claim is scored, so the test cannot be adjusted after seeing results. `mutate(target, attack)` starts from the faithful claim (the reference value rounded to six decimals at its own address) and applies one change:

- `wrong_value` keeps the address and adds 0.1 to the number, which the verifier should Contradict;
- `wrong_metric` keeps the number and moves it to the other metric's address, which should also be Contradicted;
- `numeric_string` keeps both but supplies the number as text, which should be Unscorable;
- `unsupported_repair` moves the number to an address the store does not hold, which should be Absent.

The table shows the five claims built from the contrast case together with the type of each claimed value.

In [3]:
inner=["faithful","wrong_value","wrong_metric","numeric_string","unsupported_repair"]
expected={"faithful":"Supported","wrong_value":"Contradicted","wrong_metric":"Contradicted",
          "numeric_string":"Unscorable","unsupported_repair":"Absent"}
def mutate(target,attack):
    value=round(store["facts"][target]["value"],6)
    id_,claimed=target,value
    if attack=="wrong_value":claimed=value+.1
    elif attack=="wrong_metric":id_=target.rsplit("/",1)[0]+("/contrast" if target.endswith("/brier") else "/brier")
    elif attack=="numeric_string":claimed=str(value)
    elif attack=="unsupported_repair":id_="region:experiment/predictor_repair_gain"
    return {"id":id_,"value":claimed}
display(pd.DataFrame([dict(attack=a,expected=expected[a],**(c:=mutate(outer[1],a)),value_type=type(c["value"]).__name__) for a in inner]))

,attack,expected,id,value,value_type
0,faithful,Supported,region:confirmation/reflection_73101/contrast,0.058326,float
1,wrong_value,Contradicted,region:confirmation/reflection_73101/contrast,0.158326,float
2,wrong_metric,Contradicted,region:confirmation/reflection_73101/brier,0.058326,float
3,numeric_string,Unscorable,region:confirmation/reflection_73101/contrast,0.058326,str
4,unsupported_repair,Absent,region:experiment/predictor_repair_gain,0.058326,float


**Read the output.** The rows show the attacks as the verifier will receive them. `wrong_metric` presents the correct contrast 0.058326 as the region's Brier loss, a substitution between two measurements of the same region. `numeric_string` differs from the faithful claim only in type (`str` instead of `float`), so any silent conversion of text to numbers would turn it into a false accept.

## Cross and score

The full design crosses the five attacks with the two reference cases, giving ten claims. The cell verifies each claim, asserts that its verdict equals the declared expectation and records a false accept whenever an altered claim is Supported. The table arranges the verdicts with attacks as rows and reference cases as columns, the layout of a crossed Inner by Outer Array.

In [4]:
rows=[]
for target in outer:
    for attack in inner:
        verdict=verify(store,mutate(target,attack))["verdict"]
        assert verdict==expected[attack]
        rows.append(dict(case=target,attack=attack,verdict=verdict,false_accept=attack!="faithful" and verdict=="Supported"))
design=pd.DataFrame(rows)
display(design.pivot(index="attack",columns="case",values="verdict"))

case,region:confirmation/reflection_73101/brier,region:confirmation/reflection_73101/contrast
attack,,
faithful,Supported,Supported
numeric_string,Unscorable,Unscorable
unsupported_repair,Absent,Absent
wrong_metric,Contradicted,Contradicted
wrong_value,Contradicted,Contradicted


**Read the output.** All ten claims return their declared verdicts, and the two columns are identical, so the verifier's behavior on these attacks does not depend on which metric is the target. No altered claim is Supported. The attacks were written by hand and supplied as explicit identifier and value pairs, so the design bypasses the step at which an agent would extract a claim from its own text; that step is where a live system's substitutions would occur.

## Retain the limits

A test report should state exactly which failures were exercised and which were not. The table summarizes the design by attack: the number of cases and the number of false accepts. The printed lines record that this notebook made no live agent calls and supplies no estimate of resistance to misleading prompts or of separation between conversations.

In [5]:
display(design.groupby("attack").agg(cases=("verdict","size"),false_accepts=("false_accept","sum")))
print("Live regional agent calls: 0")
print("No conversation-separation or prompt-resistance estimate is supplied.")

,cases,false_accepts
attack,,
faithful,2,0
numeric_string,2,0
unsupported_repair,2,0
wrong_metric,2,0
wrong_value,2,0


Live regional agent calls: 0
No conversation-separation or prompt-resistance estimate is supplied.


**Read the output.** Each attack was exercised on both reference cases, and false accepts are zero throughout. This is a regression result for the declared interface under constructed inputs. It does not estimate how often a language model would make these substitutions, how the verifier would perform on extracted free-text claims or how either would behave in multi-turn conversations. The live answer tests on regional facts, 70 real calls crossing four prompt rules with two source-access conditions, are inspected in the `regional_live/ch07_adversarial_prompts.ipynb` notebook of the live-run set. The chapter's multi-turn conversation experiment is recorded in the historical companion `notebooks/ch07_adversarial_prompts.ipynb`.

## What the replay establishes

A five-by-two adversarial design has been declared, crossed and scored on the regional evidence. Attacks that change the value, the metric binding, the value type or the address each produced their declared verdict for both reference cases, with no false accepts.

The design tests the checking instrument that a reporting agent would rely on, under inputs whose errors are known by construction. A live study on the same evidence would add model responses, a claim-extraction step and a scorer whose own errors must be assessed, as the chapter's conversation experiment illustrates. Chapter 8 returns to the frozen regions and examines what their confirmation measurements establish as findings.


## What to try

1. **A wrong-study attack.** Add an attack that moves the claim to `moe:confirmation/llm_compromise/worst_logloss`. Declare its expected verdict before running the design, then extend `expected` and `mutate` and rerun the crossing.
2. **A third reference case.** Add `region:confirmation/reflection_73101/share` to the Outer Array. Decide which attacks need changes to remain meaningful for a share, and explain why.
3. **A near miss.** Add an attack that changes the value by 0.0000004. Predict its verdict from the tolerance in Chapter 6 and explain whether the result is a false accept.
4. **A live design.** Describe a user prompt that tries to make an agent report the contrast as the region's Brier loss. List what a live study would need to record so that its false-accept rate could be estimated and attributed to the agent rather than the scorer.
